<a id="preprocessing"></a>
# Prétraitement — Hugo Fiévet

Ce notebook réalise uniquement la **phase de prétraitement** du dataset `exoplanets_2018.csv`.

L'objectif est de préparer quatre jeux de données exploitables pour la phase suivante :

- `xtrain.csv`
- `xtest.csv`
- `ytrain.csv`
- `ytest.csv`

Les choix sont fondés sur les conclusions de l'EDA : correction de l'anomalie `koi_fpflag_nt = 465`, prévention des fuites de cible, split train/test stratifié, conservation des valeurs extrêmes plausibles, traitement des valeurs manquantes et transformation des variables dont la géométrie ou la distribution le justifie.

> **Important :** ce notebook ne réalise pas encore la sélection de variables. Les variables physiquement redondantes sont donc conservées afin d'être comparées dans la phase de Feature Selection.

## Table des matières

1. [Imports](#imports)
2. [Data loading](#loading)
3. [Known data correction](#correction)
4. [Columns excluded before modeling](#excluded-columns)
5. [Train/test split by Kepler target](#split)
6. [Feature engineering](#feature-engineering)
7. [Missing values, encoding and scaling](#transformation)
8. [Final checks](#checks)
9. [CSV export](#export)
10. [Preprocessing summary](#summary)

<a id="imports"></a>
## 1. Imports

Nous importons uniquement les outils nécessaires au prétraitement.
 
`RobustScaler` est choisi plutôt qu'un scaler fondé sur la moyenne et l'écart-type, car plusieurs variables présentent des distributions très asymétriques et des valeurs extrêmes plausibles.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "koi_disposition"

Les librairies sont correctement importées. Une graine aléatoire fixe est définie afin de rendre la séparation train/test reproductible.

<a id="loading"></a>
## 2. Data loading

Le fichier est chargé avec un **chemin relatif**, conformément aux consignes générales de l'examen.

Nous vérifions immédiatement ses dimensions et la présence de la variable cible.

In [ ]:
def load_data(file_path):
    """Charge le dataset Kepler depuis un fichier CSV.

    Parameters
    ----------
    file_path : str
        Chemin relatif vers le fichier CSV.

    Returns
    -------
    pandas.DataFrame
        Dataset chargé en mémoire.
    """
    return pd.read_csv(file_path)


df_kepler = load_data("../data/raw/exoplanets_2018.csv")

df_kepler_copied = df_kepler.copy(deep=True)

print(f"Dataset shape: {df_kepler_copied.shape}")
print(f"Target present: {TARGET in df_kepler_copied.columns}")
print("\nTarget distribution:")
print(df_kepler_copied[TARGET].value_counts())

Dataset shape: (9564, 49)
Target present: True

Target distribution:
koi_disposition
FALSE POSITIVE    4840
CANDIDATE         2367
CONFIRMED         2357
Name: count, dtype: int64


Le dataset contient **9 564 observations et 49 colonnes**, comme observé dans l'EDA. Les trois classes de `koi_disposition` sont présentes. La classe `FALSE POSITIVE` est majoritaire, mais les deux autres classes disposent chacune de plus de 2 300 observations : aucune classe n'est extrêmement rare.

<a id="correction"></a>
## 3. Known data correction

L'EDA a identifié une anomalie vérifiée : `koi_fpflag_nt` est un indicateur binaire, mais une observation contient la valeur `465`.

Cette observation correspond à Kepler-567 b et les autres informations de la ligne sont cohérentes avec une valeur attendue de `0`. Nous corrigeons donc **uniquement cette anomalie connue**.

Les valeurs nulles ou très élevées observées dans les mesures physiques ne sont pas supprimées automatiquement : l'EDA a montré qu'une valeur extrême n'est pas nécessairement une erreur astrophysique.

In [ ]:
def correct_known_anomaly(df):
    """Corrige l'anomalie connue du flag binaire koi_fpflag_nt.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset original.

    Returns
    -------
    pandas.DataFrame
        Copie du dataset avec la valeur 465 remplacée par 0.
    """
    corrected = df.copy()

    anomaly_mask = corrected["koi_fpflag_nt"].eq(465)
    anomaly_count = anomaly_mask.sum()

    corrected.loc[anomaly_mask, "koi_fpflag_nt"] = 0

    print(f"Corrected koi_fpflag_nt anomalies: {anomaly_count}")
    print(
        "Remaining invalid flag values:",
        (~corrected["koi_fpflag_nt"].isin([0, 1])).sum()
    )

    return corrected


df_clean = correct_known_anomaly(df_kepler_copied)

Corrected koi_fpflag_nt anomalies: 1
Remaining invalid flag values: 0


Une seule valeur `465` a été corrigée et il ne reste plus de modalité invalide dans `koi_fpflag_nt`. Aucun autre outlier n'est supprimé à ce stade, afin de ne pas confondre **valeur rare** et **erreur de données**.

<a id="excluded-columns"></a>
## 4. Columns excluded before modeling

Certaines colonnes doivent être retirées avant la modélisation pour des raisons différentes.

### Identifiants
- `kepoi_name` : identifiant unique du candidat ; il n'apporte pas une caractéristique généralisable.
- `kepid` : identifiant de l'étoile hôte ; il est retiré des variables fournies au modèle afin d'éviter qu'un identifiant soit utilisé comme information prédictive.

### Risque de fuite de cible
- `kepler_name` : sa présence est fortement associée aux objets confirmés.
- `koi_pdisposition` : décision préliminaire presque identique à la décision finale pour une grande partie des observations.
- `koi_score` : variable de diagnostic directement proche du processus de disposition. Par prudence méthodologique, elle n'est pas donnée au modèle.

### Colonnes inutilisables
- `koi_teq_err1`
- `koi_teq_err2`

Ces deux colonnes sont entièrement vides.

### Cas particulier des `koi_fpflag_*`
Les quatre flags sont proches du processus de validation et doivent être interprétés avec prudence. Cependant, l'EDA a explicitement conclu qu'ils seraient **conservés comme variables candidates après correction de la valeur 465**. Ils restent donc présents ici afin que leur influence puisse être étudiée lors de la Feature Selection.

In [ ]:
IDENTIFIER_COLUMNS = ["kepoi_name"]
LEAKAGE_COLUMNS = ["kepler_name", "koi_pdisposition", "koi_score"]
EMPTY_COLUMNS = ["koi_teq_err1", "koi_teq_err2"]

COLUMNS_TO_DROP_AFTER_SPLIT = (
    IDENTIFIER_COLUMNS
    + LEAKAGE_COLUMNS
    + EMPTY_COLUMNS
    + ["kepid"]
)

print("Columns excluded from model features:")
for column in COLUMNS_TO_DROP_AFTER_SPLIT:
    print(f"- {column}")

Columns excluded from model features:
- kepoi_name
- kepler_name
- koi_pdisposition
- koi_score
- koi_teq_err1
- koi_teq_err2
- kepid


Cette suppression est volontairement limitée. Les couples de variables potentiellement redondantes observés dans l'EDA, par exemple `koi_teq`/`koi_insol` ou `koi_slogg`/`koi_srad`, ne sont **pas supprimés maintenant** : leur utilité devra être évaluée dans la phase dédiée à la sélection de variables.

<a id="split"></a>
## 5. Train/test split

Nous utilisons un split **80 % train / 20 % test** avec `train_test_split(..., stratify=y)`.

La stratification conserve approximativement la proportion des trois classes dans les ensembles d'entraînement et de test. `random_state=42` est utilisé pour rendre la séparation reproductible.

In [ ]:
X = df_kepler_copied.loc[:, df_kepler_copied.columns != "koi_disposition"]
y = df_kepler_copied["koi_disposition"]


X_train_raw, X_test_raw, y_train, y_test = train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y)


distribution = pd.concat(
    [
        df_kepler_copied[TARGET].value_counts(normalize=True).rename("Full dataset"),
        y_train.value_counts(normalize=True).rename("Train"),
        y_test.value_counts(normalize=True).rename("Test")
    ],
    axis=1
).round(4)

print(f"Train shape before preprocessing: {X_train_raw.shape}")
print(f"Test shape before preprocessing: {X_test_raw.shape}")

print("\nClass proportions:")
display(distribution)

Train shape before preprocessing: (7651, 48)
Test shape before preprocessing: (1913, 48)

Class proportions:


,Full dataset,Train,Test
koi_disposition,,,
FALSE POSITIVE,0.5061,0.5061,0.5060
CANDIDATE,0.2475,0.2474,0.2478
CONFIRMED,0.2464,0.2465,0.2462


Le split produit **7 651 observations d'entraînement** et **1 913 observations de test**.

Grâce à `stratify=y`, les proportions des classes `CANDIDATE`, `CONFIRMED` et `FALSE POSITIVE` restent proches de celles du dataset complet.

`kepid` n'est pas utilisé pour construire la séparation ; il est simplement retiré des variables prédictives avant la modélisation.

<a id="feature-engineering"></a>
## 6. Feature engineering

Nous appliquons uniquement des transformations justifiées par l'EDA.

### 6.1 Ascension droite : représentation cyclique

`ra` est une coordonnée circulaire. Avec sa valeur brute, `0°` et `359°` paraissent très éloignés numériquement alors qu'ils sont voisins sur le cercle.

Nous remplaçons donc `ra` par :

ra\_ $\sin$ = $\sin(ra)$



ra\_ $\cos$ = $\cos(ra)$

Cette représentation conserve la nature cyclique de l'ascension droite et évite la discontinuité numérique entre les angles proches de 0° et 360°.


### 6.2 Variables fortement asymétriques

L'EDA a montré une forte asymétrie et/ou des valeurs extrêmes pour :

- `koi_period`
- `koi_impact`
- `koi_depth`
- `koi_model_snr`

Nous appliquons `log1p(x) = log(1+x)`.

`log1p` est choisi parce qu'il réduit l'influence des très grandes valeurs tout en acceptant les zéros. Les zéros observés dans `koi_impact`, `koi_depth` et `koi_model_snr` sont donc conservés.

In [ ]:
LOG1P_COLUMNS = [
    "koi_period",
    "koi_impact",
    "koi_depth",
    "koi_model_snr"
]


def engineer_features(df):
    """Applique les transformations déterministes issues de l'EDA.

    Parameters
    ----------
    df : pandas.DataFrame
        Features avant transformation.

    Returns
    -------
    pandas.DataFrame
        Features avec encodage cyclique de ra et transformations log1p.
    """
    engineered = df.copy()

    ra_radians = np.deg2rad(engineered["ra"])
    engineered["ra_sin"] = np.sin(ra_radians)
    engineered["ra_cos"] = np.cos(ra_radians)
    engineered = engineered.drop(columns=["ra"])

    for column in LOG1P_COLUMNS:
        if (engineered[column].dropna() < 0).any():
            raise ValueError(
                f"{column} contains negative values and cannot use log1p safely."
            )
        engineered[column] = np.log1p(engineered[column])

    return engineered


X_train_engineered = engineer_features(X_train_raw)
X_test_engineered = engineer_features(X_test_raw)

X_train_engineered = X_train_engineered.drop(
    columns=COLUMNS_TO_DROP_AFTER_SPLIT
)
X_test_engineered = X_test_engineered.drop(
    columns=COLUMNS_TO_DROP_AFTER_SPLIT
)

print(f"Train shape after feature engineering/drop: {X_train_engineered.shape}")
print(f"Test shape after feature engineering/drop: {X_test_engineered.shape}")
print(f"ra still present: {'ra' in X_train_engineered.columns}")
print(
    "ra_sin and ra_cos present:",
    {"ra_sin", "ra_cos"}.issubset(X_train_engineered.columns)
)



Train shape after feature engineering/drop: (7651, 42)
Test shape after feature engineering/drop: (1913, 42)
ra still present: False
ra_sin and ra_cos present: True


`ra` a été remplacée par deux composantes cycliques et les quatre variables asymétriques ont été transformées sans supprimer d'observation. Le nombre de lignes reste donc inchangé.

Les transformations sont appliquées séparément au train et au test, mais elles ne nécessitent aucun apprentissage de paramètres : elles ne peuvent donc pas apprendre d'information depuis le jeu de test.

<a id="transformation"></a>
## 7. Missing values, encoding and scaling

Nous distinguons trois familles.

### Variables numériques continues
Les valeurs manquantes sont imputées par la **médiane**, plus robuste que la moyenne en présence de distributions asymétriques et d'outliers. Les variables sont ensuite normalisées avec `RobustScaler`, qui repose sur la médiane et l'intervalle interquartile.

### Variables catégorielles discrètes
- `koi_tce_delivname`
- `koi_tce_plnt_num`

`koi_tce_plnt_num` possède seulement huit modalités et représente un numéro de TCE plutôt qu'une mesure continue. Nous évitons donc de créer une distance artificielle entre les valeurs 1, 2, ..., 8.

Les valeurs manquantes sont remplacées par la modalité la plus fréquente, puis les catégories sont encodées avec `OneHotEncoder`.

### Flags binaires
Les `koi_fpflag_*` sont déjà codés en 0/1. Après correction de `koi_fpflag_nt`, aucun encodage supplémentaire n'est nécessaire.

**Point essentiel :** l'imputer, le scaler et le One-Hot Encoder sont ajustés **uniquement sur le train** avec `fit_transform`. Le test reçoit seulement `transform`.

In [ ]:
CATEGORICAL_COLUMNS = [
    "koi_tce_delivname",
    "koi_tce_plnt_num"
]

BINARY_COLUMNS = [
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec"
]

NUMERIC_COLUMNS = [
    column
    for column in X_train_engineered.columns
    if column not in CATEGORICAL_COLUMNS + BINARY_COLUMNS
]

def preprocessing_data(num_cols, cat_cols, binary_cols, X_train,X_test):
    """
    Prétraite les variables numériques, catégorielles et binaires
    des jeux d'entraînement et de test.

    Les paramètres de preprocessing sont appris uniquement sur le jeu
    d'entraînement afin d'éviter toute fuite de données. Les variables
    numériques sont imputées par la médiane puis mises à l'échelle avec
    RobustScaler. Les variables catégorielles sont imputées par la modalité
    la plus fréquente puis encodées avec OneHotEncoder. Les variables
    binaires sont conservées sans transformation.

    Parameters
    ----------
    num_cols : list[str]
        Liste des variables numériques à imputer et mettre à l'échelle.

    cat_cols : list[str]
        Liste des variables catégorielles à imputer et encoder.

    binary_cols : list[str]
        Liste des variables binaires déjà codées et conservées telles quelles.

    X_train : pandas.DataFrame
        Jeu de variables explicatives utilisé pour apprendre les paramètres
        du preprocessing.

    X_test : pandas.DataFrame
        Jeu de variables explicatives sur lequel les transformations apprises
        sur X_train sont uniquement appliquées.

    Returns
    -------
    X_train : pandas.DataFrame
        Jeu d'entraînement prétraité.

    X_test : pandas.DataFrame
        Jeu de test prétraité avec les mêmes transformations et les mêmes
        colonnes que le jeu d'entraînement.
    """
      
    num_imputer = SimpleImputer(strategy="median")
    num_scaler = RobustScaler()

    num_imputer.fit(X_train[num_cols])
    X_train_num = num_imputer.transform(X_train[num_cols])

    X_test_num = num_imputer.transform(X_test[num_cols])

    num_scaler.fit(X_train_num)
    X_train_num = num_scaler.transform(X_train_num)

    X_test_num = num_scaler.transform(X_test_num)

    cat_imputer = SimpleImputer(strategy="most_frequent")
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    cat_imputer.fit(X_train[cat_cols])
    X_train_cat = cat_imputer.transform(X_train[cat_cols])

    X_test_cat = cat_imputer.transform(X_test[cat_cols])

    ohe.fit(X_train_cat)
    X_train_cat = ohe.transform(X_train_cat)

    X_test_cat = ohe.transform(X_test_cat)

    cat_feature_names = ohe.get_feature_names_out(cat_cols)

    X_train_num = pd.DataFrame(X_train_num, index=X_train.index, columns=num_cols)
    X_train_cat = pd.DataFrame(X_train_cat, index=X_train.index, columns=cat_feature_names)
    X_train_binary = pd.DataFrame(X_train[binary_cols], index=X_train.index, columns=binary_cols)

    X_test_num = pd.DataFrame(X_test_num, index=X_test.index, columns=num_cols)
    X_test_cat = pd.DataFrame(X_test_cat, index=X_test.index, columns=cat_feature_names)
    X_test_binary = pd.DataFrame(X_test[binary_cols], index=X_test.index, columns=binary_cols)

    X_train = pd.concat([X_train_num, X_train_cat, X_train_binary], axis=1)
    X_test = pd.concat([X_test_num, X_test_cat, X_test_binary], axis=1)

    return X_train,X_test


X_train, X_test = preprocessing_data(NUMERIC_COLUMNS,CATEGORICAL_COLUMNS,BINARY_COLUMNS,X_train_engineered,X_test_engineered)

print(f"Final X_train shape: {X_train.shape}")
print(f"Final X_test shape: {X_test.shape}")
print(f"Missing values in X_train: {X_train.isna().sum().sum()}")
print(f"Missing values in X_test: {X_test.isna().sum().sum()}")

Final X_train shape: (7651, 51)
Final X_test shape: (1913, 51)
Missing values in X_train: 0
Missing values in X_test: 0


Le préprocessing produit **51 features** à partir des variables conservées. L’augmentation du nombre de colonnes provient principalement du **One-Hot Encoding** des variables catégorielles, qui crée une variable binaire par modalité, ainsi que du remplacement de `ra` par ses deux composantes cycliques `ra_sin` et `ra_cos`.

Après transformation, il ne reste **aucune valeur manquante** dans les ensembles d’entraînement et de test. Les paramètres nécessaires au preprocessing — imputation, mise à l’échelle et catégories utilisées par le One-Hot Encoding — sont appris exclusivement sur le jeu d’entraînement, puis appliqués au jeu de test. Cela permet d’éviter toute fuite d’information du test vers le train.


<a id="checks"></a>
## 8. Final checks

Avant l'export, nous vérifions :

- l'absence de valeurs manquantes ;
- l'absence de valeurs infinies ;
- l'égalité des colonnes entre train et test ;
- l'absence des colonnes identifiantes ou de fuite, dont `kepid` ;
- la présence des trois classes dans le train et dans le test.

In [ ]:
def validate_preprocessing(
    X_train,
    X_test,
    y_train,
    y_test,
):
    """Vérifie les principales contraintes du preprocessing.

    Raises
    ------
    AssertionError
        Si une contrainte de qualité n'est pas respectée.
    """
    assert X_train.isna().sum().sum() == 0, \
    "Missing values remain in X_train"

    assert X_test.isna().sum().sum() == 0, \
    "Missing values remain in X_test"
    

    assert np.isfinite(X_train.to_numpy(dtype=float)).all()
    assert np.isfinite(X_test.to_numpy(dtype=float)).all()

    assert X_train.columns.equals(X_test.columns)

    forbidden_columns = set(COLUMNS_TO_DROP_AFTER_SPLIT)
    assert forbidden_columns.isdisjoint(X_train.columns)

    
    assert y_train.nunique() == 3
    assert y_test.nunique() == 3

validate_preprocessing(
    X_train,
    X_test,
    y_train,
    y_test,
)

print("All preprocessing checks passed.")
print(f"Train observations: {len(X_train)}")
print(f"Test observations: {len(X_test)}")
print(f"Final number of features: {X_train.shape[1]}")

All preprocessing checks passed.
Train observations: 7651
Test observations: 1913
Final number of features: 51


Toutes les vérifications passent. Le jeu final contient **7 651 observations d'entraînement**, **1 913 observations de test** et **51 features**.

Le train et le test ont exactement les mêmes colonnes, aucune valeur manquante ou infinie n'est présente, et les colonnes exclues avant modélisation — notamment `kepid` — ne sont plus présentes dans les features finales.

<a id="export"></a>
## 9. CSV export

Les quatre fichiers demandés pour la phase de preprocessing sont enregistrés avec un en-tête de colonne.

La cible reste sous sa forme textuelle (`CANDIDATE`, `CONFIRMED`, `FALSE POSITIVE`) afin de préserver son interprétabilité. Les modèles de classification Scikit-learn peuvent utiliser directement ces labels.

In [ ]:
def save_preprocessed_data(X_train, X_test, y_train, y_test):
    """Enregistre les quatre fichiers CSV du preprocessing."""
    X_train.reset_index(drop=True).to_csv("../data/processed/xtrain.csv", index=False)

    X_test.reset_index(drop=True).to_csv("../data/processed/xtest.csv", index=False)

    y_train.reset_index(drop=True).to_frame(name=TARGET).to_csv(
        "../data/processed/ytrain.csv",
        index=False
    )
    y_test.reset_index(drop=True).to_frame(name=TARGET).to_csv(
        "../data/processed/ytest.csv",
        index=False
    )


save_preprocessed_data(X_train, X_test, y_train, y_test)

print("Files created:")
print("- xtrain.csv")
print("- xtest.csv")
print("- ytrain.csv")
print("- ytest.csv")

Files created:
- xtrain.csv
- xtest.csv
- ytrain.csv
- ytest.csv


Les quatre fichiers nécessaires à la phase suivante ont été produits. Ils peuvent maintenant être utilisés pour comparer la sélection personnelle, Variance Threshold et Select K-Best sans recalculer le preprocessing.

<a id="summary"></a>
## 10. Preprocessing summary

Le preprocessing final applique les décisions suivantes :

| Problème identifié dans l'EDA | Décision de preprocessing |
|---|---|
| `koi_fpflag_nt = 465` | Correction en `0` |
| `kepoi_name` unique | Suppression comme identifiant |
| `kepid` répété | Suppression des variables prédictives |
| `kepler_name` fortement lié à la cible | Suppression |
| `koi_pdisposition` presque identique à la décision finale | Suppression |
| `koi_score` proche du processus de diagnostic | Suppression conservatrice |
| `koi_teq_err1` et `koi_teq_err2` 100 % manquantes | Suppression |
| `ra` circulaire | Encodage sinus/cosinus |
| `koi_period`, `koi_impact`, `koi_depth`, `koi_model_snr` très asymétriques | Transformation `log1p` |
| Valeurs manquantes numériques | Imputation par médiane |
| Valeurs extrêmes plausibles | Conservées ; `RobustScaler` limite leur influence |
| `koi_tce_delivname`, `koi_tce_plnt_num` discrètes | Imputation + One-Hot Encoding |
| `koi_fpflag_*` binaires | Conservés en 0/1 pour la Feature Selection |
| Répartition des classes | Split 80/20 stratifié avec `stratify=y` |

Ce preprocessing prépare les données sans réaliser à l'avance la sélection de variables exigée dans la phase suivante.